# 08 — Saved Profile Reuse and Production Inference

**Author:** Sherif Abd El-Rady

A calibration profile (K, D) is **camera-specific**, not image-specific. Once computed, it applies to every image taken by that camera at the same resolution. This notebook demonstrates:

1. Loading a saved profile and validating it programmatically.
2. Applying undistortion to arbitrary new images.
3. Combining lens correction + perspective rectification in a single pipeline call.
4. Throughput benchmarking — precomputed remap tables give near-native OpenCV speed.
5. Exporting the corrected frames in a batch-processing script pattern.


In [ ]:
import sys, time
from pathlib import Path
import cv2
import numpy as np
import matplotlib.pyplot as plt

SRC = Path('..') / 'src'
if str(SRC.resolve()) not in sys.path:
    sys.path.insert(0, str(SRC.resolve()))

from calibvision.profile import CalibrationProfile
from calibvision.dewarper import LensDewarper, PerspectiveRectifier
from calibvision.utils import compute_fov

ROOT         = Path('..').resolve()
PROFILE_PATH = ROOT / 'configs' / 'saved_profiles' / 'synthetic_demo.yaml'
CALIB_DIR    = ROOT / 'data' / 'calibration_images'
print('Ready')

In [ ]:
# ── 8.1  Load and print profile ───────────────────────────────────────────────
profile = CalibrationProfile.load(PROFILE_PATH)
print(profile.summary())

# Derived camera properties
fov_x, fov_y = compute_fov(profile.camera_matrix, profile.image_size)
K = profile.camera_matrix
print(f'\nHorizontal FOV : {fov_x:.1f}°')
print(f'Vertical FOV   : {fov_y:.1f}°')
print(f'Pixel aspect   : {K[0,0]/K[1,1]:.5f}  (1.0 = square pixels)')

In [ ]:
# ── 8.2  Apply profile to new images ─────────────────────────────────────────
# Use the last 4 images as the 'new batch' (held-out from calibration set).
new_paths = sorted(CALIB_DIR.glob('*.png'))[-4:]
dewarper  = LensDewarper(profile, alpha=0.5)

fig, axes = plt.subplots(len(new_paths), 2, figsize=(13, len(new_paths) * 3))
if len(new_paths) == 1:
    axes = [axes]

for i, p in enumerate(new_paths):
    img   = cv2.imread(str(p))
    t0    = time.perf_counter()
    out   = dewarper.undistort(img)
    dt_ms = (time.perf_counter() - t0) * 1000

    axes[i][0].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    axes[i][0].set_title(f'Input: {p.name}', fontsize=8); axes[i][0].axis('off')
    axes[i][1].imshow(cv2.cvtColor(out, cv2.COLOR_BGR2RGB))
    axes[i][1].set_title(f'Undistorted  {dt_ms:.1f} ms', fontsize=8); axes[i][1].axis('off')

plt.suptitle('Profile reuse — no re-calibration required', fontsize=11)
plt.tight_layout()
plt.show()

In [ ]:
# ── 8.3  Batch throughput — how fast is the remap pipeline? ──────────────────
images = [cv2.imread(str(p)) for p in sorted(CALIB_DIR.glob('*.png'))]
_ = dewarper.undistort(images[0])  # warm-up

t0      = time.perf_counter()
results = dewarper.undistort_batch(images)
elapsed = time.perf_counter() - t0

h, w = images[0].shape[:2]
print(f'=== Batch Throughput ===')
print(f'Images : {len(images)} × {w}×{h}')
print(f'Total  : {elapsed*1000:.1f} ms')
print(f'Per img: {elapsed*1000/len(images):.2f} ms')
print(f'FPS    : {len(images)/elapsed:.0f}')
print('\nPrecomputed remap tables → O(1) initialisation cost amortised over the batch.')

In [ ]:
# ── 8.4  Full production pipeline: undistort → perspective rectify → export ───
def run_full_pipeline(
    image_path: Path,
    profile: CalibrationProfile,
    src_quad: np.ndarray | None = None,   # 4×2 float32 or None
    out_size: tuple[int,int] = (800, 600),
    alpha: float = 0.5,
) -> dict:
    """
    Full correction pipeline.

    Parameters
    ----------
    image_path  : path to the input image
    profile     : loaded CalibrationProfile
    src_quad    : four corner points [TL, TR, BR, BL] for perspective rect (optional)
    out_size    : output size for perspective rectification
    alpha       : LensDewarper alpha (0=crop, 1=keep all)

    Returns
    -------
    dict with keys: original, undistorted, rectified (None if no src_quad)
    """
    raw      = cv2.imread(str(image_path))
    dw       = LensDewarper(profile, alpha=alpha)
    undist   = dw.undistort(raw)
    rect     = None
    if src_quad is not None:
        pr   = PerspectiveRectifier(*out_size)
        rect = pr.rectify_from_clicks(undist, src_quad)
    return {'original': raw, 'undistorted': undist, 'rectified': rect}


# Demo
demo = run_full_pipeline(sorted(CALIB_DIR.glob('*.png'))[0], profile)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].imshow(cv2.cvtColor(demo['original'],   cv2.COLOR_BGR2RGB))
axes[0].set_title('Original'); axes[0].axis('off')
axes[1].imshow(cv2.cvtColor(demo['undistorted'], cv2.COLOR_BGR2RGB))
axes[1].set_title('Undistorted (alpha=0.5)'); axes[1].axis('off')
plt.suptitle('run_full_pipeline() — 10 lines end-to-end', fontsize=11)
plt.tight_layout()
plt.show()
print('✓ Full pipeline OK')

## Project 1 — Complete Summary

| Notebook | Topic | Deliverable |
|---|---|---|
| 00 | Setup & smoke test | ✓ Env verified |
| 01 | Pinhole model + distortion math | ✓ Interactive visualisations |
| 02 | Data exploration + corner detection | ✓ 20 synthetic images processed |
| 03 | Intrinsic estimation K, D | ✓ YAML profile saved |
| 04 | Lens distortion correction | ✓ Remap + alpha comparison |
| 05 | Perspective rectification | ✓ Homography + accuracy measured |
| 06 | Quality assessment | ✓ Reprojection error analysis |
| 07 | Error analysis + failure cases | ✓ N-image sensitivity, LOO |
| 08 | Profile reuse + batch inference | ✓ Batch throughput benchmarked |

**Source package:** `src/calibvision/` — importable independently of any notebook.  
**Profile:** `configs/saved_profiles/synthetic_demo.yaml` — reusable across machines.
